# Airborne Wind Energy Course Workshop 2026

During this workshop we want you to explore the quasi-steady model (QSM) framework. The QSM has already been implemented for you, so you don't need to worry about that yourself. We are only going to import some of the implemented functionalities, such that we can quickly start with getting a taste of the performance of a pumping airborne wind energy (AWE) system. Don't let all the code scare you off. We have tried to code all the examples in an intuitive manner. To do so, we have used Python, which in general is well readable and should not be too hard to understand if you have a basic understanding of programming.

First, run the following cell (click it, then Shift+Enter). It imports the packages used in this notebook and sets the plot style. It's perfectly fine if you don't understand what's happening in it.

In [ ]:
# Import required packages.
import matplotlib.pyplot as plt
import numpy as np

from qs_state_forces import solve_qs_state

# Configure the default plotting settings
size=13
params = {'legend.fontsize': 'large',
          'figure.figsize': (8,5),
          'axes.labelsize': size,
          'axes.titlesize': size,
          'xtick.labelsize': size*0.85,
          'ytick.labelsize': size*0.85,
          'axes.titlepad': 25}
plt.rcParams.update(params)

## 1. Configure environment and system properties

In this cell we set up the **wind conditions** and the **main system components** of the kite power system.  
These are collected into a single `SystemModel` object, which we will use throughout the workshop to compute the (quasi-)steady operating state of the system.

The cell does four things:

1. **Defines the wind conditions**  
   We create a `Wind` object from the dictionary `wind_props`.  
   For this workshop we use:
   - a **uniform wind field** (`wind_model = 'uniform'`), i.e. no variation with height or time,
   - a constant **reference wind speed** `speed_wind_ref` (in m/s).  

   In reality, wind speed varies with height and is turbulent in time, but here we keep it simple so we can focus on the kite dynamics.

2. **Defines the kite properties**  
   We load an aerodynamic input file with `load_aero_input()` and pass it, together with some basic geometric and mass properties, to the `Kite` class via the `kite_props` dictionary:
   - `area_wing` sets the **projected area** of the wing (in m²),
   - `mass_wing` sets the **mass** of the wing (in kg),
   - `aero_input` contains the aerodynamic coefficients and lookup data used later to compute lift and drag.

3. **Defines the tether properties**  
   We create a `RigidLumpedTether` object using the `tether_props` dictionary.  
   Here we specify:
   - the **tether diameter**,
   - the **material** (e.g. `dyneema`), which determines its stiffness and density in the model.

4. **Builds the overall system model**  
   Finally, we combine everything in a `SystemModel` object via `sys_model_props`:
   - `wind_model` → the `Wind` instance,
   - `kite` → the `Kite` instance,
   - `tether` → the `RigidLumpedTether` instance,
   - `quasi_steady = True` → we assume the system is in **quasi-steady** conditions, i.e. inertia is neglected and forces are balanced at each step.

The numbers shown in *green* in the code cell are the ones you are encouraged to change (for example, the reference wind speed or wing area).  
After modifying any of these values, **re-run the cell** so that the updated parameters are used in the rest of the notebook.

In [ ]:
# Create an object setting the wind conditions.
from tether import RigidLumpedTether
from utils import load_aero_input
from wind import Wind
from system_model import SystemModel
from kite import Kite

# Create an object setting the wind conditions.
wind_props = {
    'speed_wind_ref': 10.,  # Reference wind speed [m/s]
    'wind_model': 'uniform', # Wind model type
}
wind = Wind(**wind_props)

# Create an object setting the system properties of the kite power system.
aero_input = load_aero_input() # Load the aerodynamic input data for the TUDELFT V3 kite.
kite_props = {
    'mass_wing': 0,  # [kg]
    'area_wing': 19.75,  # [m^2]
    'aero_input': aero_input,

}
kite = Kite(**kite_props)

# Create an object setting the system properties of the tether.
tether_props = {
    'diameter': 1e-12,  # [m]
    'material': 'dyneema',
}
tether = RigidLumpedTether(**tether_props)

# Create the system model object.
sys_model_props = {
    'wind_model': wind,
    'kite': kite,
    'tether': tether,
    'quasi_steady': True
}
sys_model = SystemModel(**sys_model_props)

## 2. Verifying Loyd's theory
As has been addressed in the course, the QSM represents the kite as a point mass. Next, we'll use the model to evaluate how sensitive the power output is to the postion of the kite.

In the following plots, the angle of attack has been included in the QSM, such that the effect of weight, actuation and position on the trim condition is accounted for. 

You can use the following plot to explore the influence of different parameters in the performance and kinematics of the kite.

In [ ]:
# Interactive quasi-steady forces UI
# Encapsulated in ui_qs.launch_qs_forces_ui to keep notebook clean.
from ui_qs import launch_qs_forces_ui

launch_qs_forces_ui(
    sys_model=sys_model,
    aero_input=aero_input,
    kite=kite,
    tether=tether,
    course_deg=90,
    mass_wing=0.0,
    elevation_deg=0.0,
    azimuth_deg=0.0,
    wind_speed=9.0,
    speed_radial=0.0,
    depower=0.0,
    continuous_update=True,  # redraw while dragging; set False to redraw only on release
)

**MASSLESS KITE**

**Q1** What is the optimal reelout speed at a wind speed of 9ms at perfect crosswind? What happens to the optimal reelout speed if we increase elevation/azimuth?

**Q2** Does the radial distance affect the power? Why? Hint: look at the system properties.

**Q3** Does the apparent wind speed depend on the course angle for a massless kite? 

**Q4** What is the azimuth where the power is maximum? And the tangential speed?

**MASS ADDITION**

**Q1-4** Answer to the previous questions for a kite with mass of 40 kg and a wind speed of 12 m/s

**Q5** What is the elevation that leads to a static flight condition? And if we depower the kite (depower = 1)?

**Q6** If I reel-in at 3 m/s, what will happen to the tangential speed? At which elevation will the kite fly in radial flight?

## 3. Relation between reel-out power and factor
To get a better idea of how the reel-out power changes with the reel-out factor, in the next code block we're going to iterate over a range of reel-out factors (`reel_out_factors`) and calculate the power output for each in the code block below. We write the results to the `power` array, which we use to plot the reel-out power against the reel-out factor.   

In [ ]:
# Create an object containing the kite position and course angle.
kite_state = {
    'distance_radial': 200.,  # [m]
    'angle_azimuth': 0.*np.pi/180.,  # [rad]
    'angle_elevation': 0.*np.pi/180.,  # [rad]
    'angle_course': 90.*np.pi/180.,  # [rad]
    'input_depower': 0.0,  # [-]
}

# Set the range of reeling factors for which we will determine the steady states.
n_reeling_factors = 101
reel_out_factors = np.linspace(0.0, 0.5, n_reeling_factors)

# Create an array to which we will save our results.
power = np.zeros(n_reeling_factors)

# Iterate over the reel-out factors.
for i, rf in enumerate(reel_out_factors):
    kite_state["speed_radial"] = sys_model.wind.speed_wind_ref * rf
    try:
        qs_state = solve_qs_state(
                model=sys_model,
                state=kite_state,
            )
            
        power[i] = qs_state.tension_tether_ground * qs_state.speed_radial  
    except:
        power[i] = np.nan
    # print("Tension tether ground:", qs_state["tension_tether_ground"])
print("Maximum reel-out power: {:.1f}W".format(np.nanmax(power)))

# f_opt_spherical = 1/3. * np.cos(kite_state["angle_elevation"]) * np.cos(kite_state["angle_azimuth"])
# plt.axvline(f_opt_spherical, linestyle='--', color='r')

# Plot the results.
plt.plot(reel_out_factors, power)
plt.axvline(1/3., linestyle='--', color='k')
plt.xlabel('Reel-out factor [-]')
plt.ylabel('Reel-out power [W]')
plt.grid()

So indeed, from the upper plot we can observe that the power is maximum for a reeling factor of 1/3. How does this change when we change the position of the kite? Play around with the position of the kite in the previous code block and see how the curve changes.

**Q7** What is the optimal reel-out factor when flying at an elevation angle of 𝛽 = 30°, an azimuth angle of ɸ = 15°, a course angle of 𝛸 = 100°?

## 4. Pumping cycle performance
The `solve_qs_state` calculation only determines the instaneous performance (for a single point in time) during reel-out. For now we assume that evaluating a single point already gives us a reasonable representation of the whole reel-out phase. However, we also need to consider the reel-in phase to evaluate the whole pumping cycle of a pumping AWE system. Do you think including the reel-in phase affects the optimal reel-out factor? Let's solve two state instances: one that represents the reel-out phase (`kite_state_out`) and one that represents the reel-in phase (`kite_state_in`). In the lower code block, we declare a new function `calc_mean_cycle_power` to group all the required steps to calculate the mean cycle power together. By introducing this layer of abstraction, we can perform the calculation using a simple line of code later on in the notebook and at that point we don't need to worry about the details of the calculation anymore. Also, this makes it really easy to re-do the calculation using a slightly different input.

Simply run the lower code block to declare the function that we'll use later on.

In [ ]:
# Create objects containing the kite position and course angle for reel-out and reel-in.
kite_state_out = {
    'distance_radial': 200.,  # [m]
    'angle_azimuth': 0.,  # [rad]
    'angle_elevation': np.radians(0),  # [rad]
    'angle_course': np.radians(90),  # [rad]
    'input_depower': 0.0,  # [-]
}

kite_state_in = {
    'distance_radial': 200.,  # [m]
    'angle_azimuth': 0.,  # [rad]
    'angle_elevation': 0.*np.pi/180.,  # [rad]
    'angle_course': 0.*np.pi/180.,  # [rad]
    'input_depower': 0.0,  # [-]
}

# Create a function for calculating the mean cycle power given the reeling factor during reel-out and reel-in.
def calc_mean_cycle_power(kite_state_out, kite_state_in):
    try:
        # Determine steady states for reel-out and reel-in.
        qs_state_out = solve_qs_state(model=sys_model, state=kite_state_out)
        qs_state_in = solve_qs_state(model=sys_model, state=kite_state_in)
    except RuntimeError:
        # No steady flight is possible for these settings (e.g. the kite is too heavy for this wind).
        # Return NaN (not a number): it shows up as a gap in the plots.
        return np.nan, np.nan, np.nan

    p_out = qs_state_out.tension_tether_ground * qs_state_out.speed_radial
    v_out = qs_state_out.speed_radial
    p_in = qs_state_in.tension_tether_ground * qs_state_in.speed_radial
    v_in = -qs_state_in.speed_radial

    # Assuming that the reeling distance is the same for the reel-out and reel-in: the fraction of time spent per
    # phase can be calculated from the reeling speeds. 
    mean_cycle_power = (p_out/v_out + p_in/v_in) * v_out*v_in/(v_out + v_in)
    
    return mean_cycle_power, p_out, p_in

**Q8** How do we differentiate between the the state of the kite when reeling-out with respect to reeling-in?

Again, we're going to iterate over a range of reel-out factors (`reel_out_factors`), while fixing the reel-in factor (`reel_in_factor`) to a value of -0.2. Addtional to the reel-out power, this time we also calculate the reel-in and mean cycle power and plot them against the reel-out factor.

In [ ]:
# Set the range of reeling factors for which we will determine the steady states.
n_reeling_factors = 100
reel_out_factors = np.linspace(0.05, 0.4, n_reeling_factors)
reel_in_factor = -0.2

# Create arrays to which we will save our results.
mean_cycle_power = np.zeros(n_reeling_factors)
power_out = np.zeros(n_reeling_factors)
power_in = np.zeros(n_reeling_factors)

# Iterate over the reel-out factors and calculate corresponding mean cycle power.
for i, rf_out in enumerate(reel_out_factors):
    kite_state_out["speed_radial"] = sys_model.wind.speed_wind_ref * rf_out
    kite_state_in["speed_radial"] = sys_model.wind.speed_wind_ref * reel_in_factor
    mcp, p_out, p_in = calc_mean_cycle_power(kite_state_out, kite_state_in)
    mean_cycle_power[i] = mcp
    power_out[i] = p_out
    power_in[i] = p_in

# Plot the results.
fig, ax = plt.subplots(3, 1, figsize=(8,8))
ax[0].plot(reel_out_factors, mean_cycle_power*1e-3)
ax[0].set_ylabel('Mean cycle\npower [kW]')
ax[1].plot(reel_out_factors, power_out*1e-3)
ax[1].set_ylabel('Reel-out\npower [kW]')
ax[2].plot(reel_out_factors, power_in*1e-3)
ax[2].set_ylabel('Reel-in\npower [kW]')

ax[1].axvline(1/3., linestyle='--', color='k')
ax[-1].set_xlabel('Reel-out factor [-]')
for a in ax:
    a.grid()

It turns out that the mean cycle power is negative and monotonically decreasing. This implies that we are consuming energy regardless of the choice for the reel-out factor. 

**Q9** Why do you think we're not producing energy for the given example? Hint: think about the state of the kite.

## 5. Generating positive net power output
Generally the kite is de-powered during the rectraction phase: the aerodynamics properties of the kite are changed such that less energy needs to be put into the system to lower the kite. Let's try again to identify the optimal reel-out factor, however, this time we differentiate between the aerodynamic properties during reel-out and reel-in. To do so, we increase the `input_depower`. As we saw in the first plots, this decreases the lift coefficient for the reel-in phase.

In [ ]:
# Set the range of reeling factors for which we will determine the steady states.
n_reeling_factors = 100
reel_out_factors = np.linspace(0.05, 0.4, n_reeling_factors)
reel_in_factor = -0.2

kite_state_in["input_depower"] = 1

# Create arrays to which we will save our results.
mean_cycle_power = np.zeros(n_reeling_factors)
power_out = np.zeros(n_reeling_factors)
power_in = np.zeros(n_reeling_factors)

# Iterate over the reel-out factors and calculate corresponding mean cycle power.
for i, rf_out in enumerate(reel_out_factors):
    kite_state_out["speed_radial"] = sys_model.wind.speed_wind_ref * rf_out
    kite_state_in["speed_radial"] = sys_model.wind.speed_wind_ref * reel_in_factor
    mcp, p_out, p_in = calc_mean_cycle_power(kite_state_out, kite_state_in)
    mean_cycle_power[i] = mcp
    power_out[i] = p_out
    power_in[i] = p_in

# Plot the results.
fig, ax = plt.subplots(3, 1, figsize=(8,8))
ax[0].plot(reel_out_factors, mean_cycle_power*1e-3)
ax[0].set_ylabel('Mean cycle\npower [kW]')
ax[1].plot(reel_out_factors, power_out*1e-3)
ax[1].set_ylabel('Reel-out\npower [kW]')
ax[2].plot(reel_out_factors, power_in*1e-3)
ax[2].set_ylabel('Reel-in\npower [kW]')

ax[1].axvline(1/3., linestyle='--', color='k')
ax[-1].set_xlabel('Reel-out factor [-]')
for a in ax:
    a.grid()

Nice! We were able to reduce the reel-in power substantially. Now we are actually producing energy on average over the full range of reel-out factors. The mean cycle power curve shows a maximum at a different reel-out factor than the reel-out power curve. 

**Q10** Is it beneficial to reel-out slower or faster compared to the optimal condition that follows from Loyd's theory?

**Q11** In the previous code block, change the reel-in factor. How does this change the optimal reel-out factor? 

## 6. Optimal reel-out and reel-in factors
The optimal reel-out factor is also dependent on the reel-in factor. Let's try to find the combination of reel-out and reel-in factors that maximise the mean cycle power using SciPy's `minimize` function that is part of the optimization package. As can be read in [SciPy's documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html), `minimize` at least needs: the objective function to be minimized and an initial guess for the optimal solution. Note that maximizing the mean cycle power is the same as minimizing the negated mean cycle power. For a reel-in factor of -0.2, we see from the previous figure that a reel-out factor of roughly 0.15 maximizes the mean cycle power: so let's use those values for our intial guess. 

In [ ]:
from scipy.optimize import minimize

# Define the function that is used for evaluating the objective in the optimization.
def objective_function(x):
    # The decision vector x, over which is minimized, consists of the reel-out and reel-in factor.
    rf_out = x[0]
    rf_in = x[1]
    kite_state_out["speed_radial"] = sys_model.wind.speed_wind_ref * rf_out
    kite_state_in["speed_radial"] = sys_model.wind.speed_wind_ref * rf_in
    mcp = calc_mean_cycle_power(kite_state_out=kite_state_out, kite_state_in=kite_state_in)[0]  # Call the previously defined function.
    if np.isnan(mcp):
        return 1e6  # No steady flight here: a large penalty steers the optimizer away.
    return -mcp  # Maximizing the mean cycle power is equivalent to minimizing its negate.

# Configure optimization problem.
starting_point = (0.15, -0.2)  # Initial values of the reel-out and reel-in factor, respectively.
bounds = (
    (0.01, 0.4),  # Lower and upper bound for the reel-out factor.
    (-.4, -.01),  # Lower and upper bound for the reel-in factor.
)
# Perform optimization.
optimization_result = minimize(objective_function, starting_point, bounds=bounds, method='SLSQP')  
print(optimization_result)  # Print optimization results to the screen.

**Q12** How much differts the newly found optimal reel-out factor with respect to the one that we found for a fixed reel-in factor? Hint: you can find the optimal value in `optimization_result.x`?

## 7. Verifying the optimal solution
Let's verify if the reel-out and reel-in factor combination found by the optimization indeed corresponds to a maximum of the mean cycle power. This time we are going to evaluate the mean cycle power for a range of reel-out factors, but also reel-in factors. Is the mean cycle power more sensitive to the reel-out or reel-in factor?

In [ ]:
# Get the optimal reel-out and reel-in factor from the optimal point that we found above.
rf_out_opt = optimization_result['x'][0]
rf_in_opt = optimization_result['x'][1]

# Vary the reel-out factor around the optimal point.
n_reeling_factors = 100
reel_out_factors = np.linspace(0.05, 0.5, n_reeling_factors)

# Create array to which we will save our results.
mean_cycle_power = np.zeros(n_reeling_factors)

# Iterate over the reel-out factors and calculate corresponding mean cycle power.
for i, rf_out in enumerate(reel_out_factors):
    kite_state_out["speed_radial"] = sys_model.wind.speed_wind_ref * rf_out
    kite_state_in["speed_radial"] = sys_model.wind.speed_wind_ref * rf_in_opt
    mean_cycle_power[i] = calc_mean_cycle_power(kite_state_out=kite_state_out, kite_state_in=kite_state_in)[0]

# Plot the mean cycle power w.r.t. the reel-out factor.
plt.figure(figsize=(7,4))
plt.plot(reel_out_factors, mean_cycle_power, label="cycle")
plt.axvline(rf_out_opt, linestyle='--', color='g')  # Plot a vertical, dashed line at the optimal point.
plt.xlabel('Reel-out factor [-]')
plt.ylabel('Power [W]')
plt.legend()
plt.grid()

# Vary the reel-in factor around the optimal point.
reel_in_factors = np.linspace(-1., -0.01, n_reeling_factors)

# Create array to which we will save our results.
mean_cycle_power = np.zeros(n_reeling_factors)

# Iterate over the reel-in factors and calculate corresponding mean cycle power.
for i, rf_in in enumerate(reel_in_factors):
    kite_state_out["speed_radial"] = sys_model.wind.speed_wind_ref * rf_out_opt
    kite_state_in["speed_radial"] = sys_model.wind.speed_wind_ref * rf_in
    mean_cycle_power[i] = calc_mean_cycle_power(kite_state_out=kite_state_out, kite_state_in=kite_state_in)[0]

# Plot the mean cycle power w.r.t. the reel-in factor.
plt.figure(figsize=(7,4))
plt.plot(reel_in_factors, mean_cycle_power, label="cycle")
plt.axvline(rf_in_opt, linestyle='--', color='g')  # Plot a vertical, dashed line at the optimal point.
plt.xlabel('Reel-in factor [-]')
plt.ylabel('Power [W]')
plt.legend()
plt.grid()

## 8. Ideal power curve
So far we have only assessed the system's performance for a single wind speed. However, as you probably know, the performance is highly dependent on the wind speed. The power in the wind is a function of the wind speed cubed. Do you think that the power we can harness using an AWE system is also a function of the wind speed cubed?

In [ ]:
# Define a function for determining cycle performance indicators not only as a function of the reeling factors, but
# also as a function of the wind speed.
def calc_mean_cycle_power_region1(rf_out, rf_in, v_wind):
    # We can re-use the wind conditions object that we created at the start by updating the wind speed.
    wind = Wind(speed_wind_ref=v_wind, wind_model='uniform')
    sys_model = SystemModel(
        wind_model=wind,
        kite=kite,
        tether=tether,
        quasi_steady=True
    )
    # Determine steady states for reel-out and reel-in.
    kite_state_out["speed_radial"] = sys_model.wind.speed_wind_ref * rf_out
    kite_state_in["speed_radial"] = sys_model.wind.speed_wind_ref * rf_in
    try:
        qs_state_out = solve_qs_state(model=sys_model, state=kite_state_out)
        qs_state_in = solve_qs_state(model=sys_model, state=kite_state_in)
    except RuntimeError:
        # No steady flight is possible here (e.g. the kite is too heavy for this wind): return NaN (not a number).
        return {key: np.nan for key in ['mean_cycle_power', 'power_reel_out', 'tether_force_out', 'speed_reel_out',
                                        'power_reel_in', 'tether_force_in', 'speed_reel_in']}

    p_out = qs_state_out.tension_tether_ground * qs_state_out.speed_radial
    v_out = qs_state_out.speed_radial
    p_in = qs_state_in.tension_tether_ground * qs_state_in.speed_radial
    v_in = -qs_state_in.speed_radial

    # Calculate mean cycle power from reel-out and reel-in steady state solutions.
    mean_cycle_power = (p_out/v_out + p_in/v_in) * v_out*v_in/(v_out + v_in)
    
    # Return a dictionary object with the results.
    result = {
        'mean_cycle_power': mean_cycle_power,
        'power_reel_out': p_out,
        'tether_force_out': qs_state_out.tension_tether_ground,
        'speed_reel_out': v_out,
        'power_reel_in': p_in,
        'tether_force_in': qs_state_in.tension_tether_ground,
        'speed_reel_in': v_in,
    }
    return result

# Define a new objective function that uses the decision vector and the wind speed as input.
def obj_fun_region1(x, v_wind):
    mcp = calc_mean_cycle_power_region1(x[0], x[1], v_wind)['mean_cycle_power']
    if np.isnan(mcp):
        return 1e6  # No steady flight here: a large penalty steers the optimizer away.
    return -mcp

# Configure optimization problem.
starting_point = (0.2, -0.2)
bounds = ((.01, 0.4), (-0.3, -.01))

# Repeat optimizations for a range of wind speeds.
n_wind_speeds = 14
winds_speeds = np.linspace(7, 20, n_wind_speeds)

# Create arrays to which we will save our results.
mean_cycle_power = np.full(n_wind_speeds, np.nan)
power_out, force_out, speed_out = np.full(n_wind_speeds, np.nan), np.full(n_wind_speeds, np.nan), np.full(n_wind_speeds, np.nan)
power_in, force_in, speed_in = np.full(n_wind_speeds, np.nan), np.full(n_wind_speeds, np.nan), np.full(n_wind_speeds, np.nan)

# Iterate over the range of wind speeds.
for i, v_wind in enumerate(winds_speeds):
    # Perform optimization - wind speed is passed as an argument.
    res = minimize(obj_fun_region1, starting_point, bounds=bounds, method='SLSQP', args=(v_wind,))
    
    # Get the result dictionary for the optimal point and write to the result arrays.
    performance_dict = calc_mean_cycle_power_region1(res['x'][0], res['x'][1], v_wind)
    mean_cycle_power[i] = performance_dict['mean_cycle_power']
    power_out[i] = performance_dict['power_reel_out']
    force_out[i] = performance_dict['tether_force_out']
    speed_out[i] = performance_dict['speed_reel_out']
    power_in[i] = performance_dict['power_reel_in']
    force_in[i] = performance_dict['tether_force_in']
    speed_in[i] = performance_dict['speed_reel_in']
    if np.isnan(performance_dict['mean_cycle_power']):
        print(f"No steady flight possible at {v_wind:.0f} m/s: this wind speed is skipped.")
    else:
        starting_point = res['x']

# Plot specifics on the optimal performance w.r.t. the wind speed.
fig, ax = plt.subplots(3, 1, sharex=True, figsize=(8,8))    
ax[0].plot(winds_speeds, mean_cycle_power*1e-3, 's', label="cycle")
ax[0].plot(winds_speeds, power_out*1e-3, label="out")
ax[0].plot(winds_speeds, -power_in*1e-3, label="-in")
ax[0].legend()
ax[1].plot(winds_speeds, force_out*1e-3, label="out")
ax[1].plot(winds_speeds, force_in*1e-3, label="in")
ax[1].legend()
ax[2].plot(winds_speeds, speed_out, label="out")
ax[2].plot(winds_speeds, speed_in, label="-in")
ax[2].legend()

ax[0].set_ylabel('Power [kW]')
ax[1].set_ylabel('Force [kN]')
ax[2].set_ylabel('Reeling speed [m/s]')
ax[2].set_xlabel('Wind speed [m/s]')

for a in ax: a.grid()
    
# z = np.polyfit(winds_speeds[10:20:3], mean_cycle_power[10:20:3]*1e-3, 3)
# p = np.polyval(z, winds_speeds)
# ax[0].plot(winds_speeds, p, 's', label="cubic")

Is the power curve that we obtained realistic? What happens if we evaluate the power for even higher wind speeds? Try it out by changing the maximum value for the `winds_speeds` array. Probably the curve is not really representative for an actual system: we need to consider some system limits, such as maximum force that the tether can hold and maximum power for which the generator is designed.

## 9. Realistic power curve
In the previous code block we have created a function for determining the mean cycle power for the case in which no limits are active on the system, which represents the first region of our power curve. Now let's introduce a second and third region: where the maximum tether force and both the maximum tether force and power is reached.

We also make the system realistic: the TU Delft V3 kite, as flown on the university's 20 kW demonstrator. The kite now has a mass (wing, bridle lines and the kite control unit, KCU), the tether is 4 mm thick and has drag, and the ground station limits the tether force to 8.4 kN and the generator power to 20 kW. During reel-in the kite flies depowered at 50° elevation, where it pulls less on the tether.

Note that this cell replaces the massless kite and drag-free tether of Section 1, and the reel-in elevation of Section 4. To go back, re-run Sections 1 and 4.

In [ ]:
# The V3 kite on the TU Delft 20 kW demonstrator: realistic mass, tether and system limits.
# This replaces the massless kite and drag-free tether of Section 1 (re-run Section 1 to go back).
kite = Kite(
    mass_wing=13.6,  # Wing and bridle lines [kg]
    mass_kcu=8.4,  # Kite control unit, hanging below the wing [kg]
    area_wing=19.75,  # [m^2]
    aero_input=aero_input,
)
tether = RigidLumpedTether(diameter=0.004, material='dyneema')  # 4 mm Dyneema tether

# System limits.
tether_force_limit = 8400  # Maximum tether force [N]
power_limit = 20000  # Generator power [W]

# Reel-in: the depowered kite flies at a higher elevation, where it pulls less on the tether.
kite_state_in["angle_elevation"] = np.radians(50)

# Determine cycle performance indicators for the maximum tether force limit case. The reel-out factor is set by the
# tether force limit and is not part of the decision variable anymore.
def calc_mean_cycle_power_region2(rf_in, v_wind):
    # Set wind speed.
    wind = Wind(speed_wind_ref=v_wind, wind_model='uniform')
    sys_model = SystemModel(
        wind_model=wind,
        kite=kite,
        tether=tether,
        quasi_steady=True
    )
    
    kite_state_out["tension_tether_ground"] = tether_force_limit
    kite_state_out["speed_tangential"] = 3 * v_wind  # Initial guess for the solver
    kite_state_in["speed_radial"] = sys_model.wind.speed_wind_ref * rf_in
    try:
        # The tether force limit fully specifies the reel-out state: now the reel-out speed is unknown.
        qs_state_out = solve_qs_state(
            model=sys_model,
            state=kite_state_out,
            unknown_vars=['speed_radial', 'speed_tangential', 'input_steering'],
        )
        # Determine steady state for reel-in.
        qs_state_in = solve_qs_state(model=sys_model, state=kite_state_in)
    except RuntimeError:
        # No steady flight is possible here (e.g. the kite is too heavy for this wind): return NaN (not a number).
        return {key: np.nan for key in ['mean_cycle_power', 'power_reel_out', 'tether_force_out', 'speed_reel_out',
                                        'power_reel_in', 'tether_force_in', 'speed_reel_in']}

    p_out = qs_state_out.tension_tether_ground * qs_state_out.speed_radial
    v_out = qs_state_out.speed_radial
    p_in = qs_state_in.tension_tether_ground * qs_state_in.speed_radial
    v_in = -qs_state_in.speed_radial

    # Calculate mean cycle power from reel-out and reel-in steady state solutions.
    mean_cycle_power = (p_out/v_out + p_in/v_in) * v_out*v_in/(v_out + v_in)
    
    # Return a dictionary object with the results.
    result = {
        'mean_cycle_power': mean_cycle_power,
        'power_reel_out': p_out,
        'tether_force_out': tether_force_limit,
        'speed_reel_out': v_out,
        'power_reel_in': p_in,
        'tether_force_in': qs_state_in.tension_tether_ground,
        'speed_reel_in': v_in,
    }
    return result

# Define the objective function for region 2.
def obj_fun_region2(x, v_wind):
    mcp = calc_mean_cycle_power_region2(x[0], v_wind)['mean_cycle_power']
    if np.isnan(mcp):
        return 1e6  # No steady flight here: a large penalty steers the optimizer away.
    return -mcp

# Determine cycle performance indicators for the maximum tether force and power limit case.
def calc_mean_cycle_power_region3(rf_in, v_wind):
    # Set wind speed.
    wind = Wind(speed_wind_ref=v_wind, wind_model='uniform')
    sys_model = SystemModel(
        wind_model=wind,
        kite=kite,
        tether=tether,
        quasi_steady=True
    )
    
    # The tether force and power limit fully specify the reel-out parameters and are independent of the wind speed.
    p_out = power_limit
    f_out = tether_force_limit
    v_out = p_out/f_out

    # Determine steady state for reel-in.
    kite_state_in["speed_radial"] = sys_model.wind.speed_wind_ref * rf_in
    try:
        qs_state_in = solve_qs_state(model=sys_model, state=kite_state_in)
    except RuntimeError:
        # No steady flight is possible here (e.g. the kite is too heavy for this wind): return NaN (not a number).
        return {key: np.nan for key in ['mean_cycle_power', 'power_reel_out', 'tether_force_out', 'speed_reel_out',
                                        'power_reel_in', 'tether_force_in', 'speed_reel_in']}
    p_in = qs_state_in.tension_tether_ground * qs_state_in.speed_radial
    v_in = -qs_state_in.speed_radial

    # Calculate mean cycle power from reel-out and reel-in steady state solutions.
    mean_cycle_power = (p_out/v_out + p_in/v_in) * v_out*v_in/(v_out + v_in)
    
    # Return a dictionary object with the results.
    result = {
        'mean_cycle_power': mean_cycle_power,
        'power_reel_out': p_out,
        'tether_force_out': f_out,
        'speed_reel_out': v_out,
        'power_reel_in': p_in,
        'tether_force_in': qs_state_in.tension_tether_ground,
        'speed_reel_in': v_in,
    }
    return result

# Define the objective function for region 3.
def obj_fun_region3(x, v_wind):
    mcp = calc_mean_cycle_power_region3(x[0], v_wind)['mean_cycle_power']
    if np.isnan(mcp):
        return 1e6  # No steady flight here: a large penalty steers the optimizer away.
    return -mcp

# Configure optimization problem.
starting_point = (0.2, -0.2)

# Repeat optimizations for a range of wind speeds.
n_wind_speeds = 14
winds_speeds = np.linspace(7, 20, n_wind_speeds)

# Create arrays to which we will save our results.
# They start as NaN (not a number): a wind speed without a solution leaves a gap in the plot.
mean_cycle_power = np.full(n_wind_speeds, np.nan)
power_out, force_out, speed_out = np.full(n_wind_speeds, np.nan), np.full(n_wind_speeds, np.nan), np.full(n_wind_speeds, np.nan)
power_in, force_in, speed_in = np.full(n_wind_speeds, np.nan), np.full(n_wind_speeds, np.nan), np.full(n_wind_speeds, np.nan)

i_region = 0  # Counter for tracking to which region the evaluated wind speed belongs.
for i, v_wind in enumerate(winds_speeds):
    # Check which region is active, perform the relevant optimization, and get the result dictionary for the
    # optimal point.
    if i_region == 0:
        bounds = ((.01, 0.3), (-0.6, -0.01))
        res = minimize(obj_fun_region1, starting_point, bounds=bounds, method='SLSQP', args=(v_wind,))
        performance_dict = calc_mean_cycle_power_region1(res['x'][0], res['x'][1], v_wind)
        # If the tether force limit is exceeded, proceed to the next region.
        if performance_dict['tether_force_out'] > tether_force_limit:
            i_region = 1
            starting_point = (res['x'][1],)
    if i_region == 1:
        bounds = ((-2, -0.01),)
        res = minimize(obj_fun_region2, starting_point, bounds=bounds, method='SLSQP', args=(v_wind,))
        performance_dict = calc_mean_cycle_power_region2(res['x'][0], v_wind)
        # If the power limit is exceeded, proceed to the next region.
        if performance_dict['power_reel_out'] > power_limit:
            i_region = 2
            starting_point = (res['x'][0],)
    if i_region == 2:
        res = minimize(obj_fun_region3, starting_point, bounds=bounds, method='SLSQP', args=(v_wind,))
        performance_dict = calc_mean_cycle_power_region3(res['x'][0], v_wind)

    # Update the result arrays.
    mean_cycle_power[i] = performance_dict['mean_cycle_power']
    power_out[i] = performance_dict['power_reel_out']
    force_out[i] = performance_dict['tether_force_out']
    speed_out[i] = performance_dict['speed_reel_out']
    power_in[i] = performance_dict['power_reel_in']
    force_in[i] = performance_dict['tether_force_in']
    speed_in[i] = performance_dict['speed_reel_in']
    if np.isnan(performance_dict['mean_cycle_power']):
        print(f"No steady flight possible at {v_wind:.0f} m/s: this wind speed is skipped.")
    else:
        starting_point = res['x']

# Plot power curve.
fig, ax = plt.subplots(3, 1, sharex=True, figsize=(8,8))   
ax[0].plot(winds_speeds, mean_cycle_power*1e-3, label="cycle")
ax[0].plot(winds_speeds, power_out*1e-3, label="out")
ax[0].plot(winds_speeds, -power_in*1e-3, label="-in")
ax[0].legend()
ax[1].plot(winds_speeds, force_out*1e-3, label="out")
ax[1].plot(winds_speeds, force_in*1e-3, label="in")
ax[1].legend()
ax[2].plot(winds_speeds, speed_out, label="out")
ax[2].plot(winds_speeds, speed_in, label="-in")
ax[2].legend()

ax[0].set_ylabel('Power [kW]')
ax[1].set_ylabel('Force [kN]')
ax[2].set_ylabel('Reeling speed\n[m/s]')
ax[2].set_xlabel('Wind speed [m/s]')

for a in ax: a.grid()

**Q13** Identify the three regions in the power curve. At which wind speeds is the tether force limit reached, and at which the power limit?

**Q14** Above about 11 m/s the mean cycle power drops, although the reel-out power stays at its limit. Look at the reel-in force: why does this happen? What could the kite do differently during reel-in at high wind speeds? Try changing the reel-in elevation in the previous cell (it fails if the kite is too heavy to fly that high in light wind).

Want to go further? The model in this notebook is a strongly simplified version of an early form of the model in [AWETrim](https://awegroup.github.io/AWETrim/). AWETrim goes much further: the deformation of the kite, optimised flight paths and comparison with flight data.

Thank you for joining the workshop, I hope that you enjoyed it! Please take a minute to give feedback on the workshop. [Click here](https://docs.google.com/forms/d/e/1FAIpQLSen-pcbHG2a4kls6EONPkKxtgDshKIwKU7EY6aJ4BWSPtfXJA/viewform?usp=sf_link "Open feedback form") to open the feedback form.

## 10. Extra: more detailed flight
In case you have some time left and are still interested: in the lower block you can play around with resolving the flight trajectory of the kite with a higher number of time points.

In [ ]:
from reelout import simulate_pattern
import numpy as np


PHYSICAL_CONFIG = {
    "mass_wing": 15,
    "area_wing": 19.75,
    "tether_diameter": 0.001,
    "mass_kcu":0,
}

# PATH_PARAMETERS = {
#     "pattern_type": "cst_helix", # "cst_lissajous" or    "r0": 200,
#     "az_amp0": np.radians(20),
#     "beta_amp0": np.radians(20),
#     "beta0": np.radians(25),
#     # "downloops": True,
#     "distance_radial_start": 200,
#     # "downloops": True,    # Only for Lissajous
# }

PATH_PARAMETERS = {
    "pattern_type": "cst_lissajous", # "cst_list_helix"
    "r0": 200,
    "az_amp0": np.radians(20),
    "beta_amp0": np.radians(5),
    "beta0": np.radians(25),
    # "downloops": True,
    "distance_radial_start": 200,
    # "downloops": True,    # Only for Lissajous
}

RADIAL_PARAMETERS = {
    "reeling_strategy": "constant",  # "constant" (fixed reeling speed) or "force" (speed follows the tether force)
    "reeling_speed": 2,  # m/s, only for "constant"
    "slope_winch_force": 500,  # N/(m/s)^2, only for "force": tether force = slope * reeling speed^2
}

WIND_CONFIG = {
    "speed_wind_at_100": 10,
    "z0": 0.002,
    "model_type": "uniform",
}

variables_to_plot = [
    "tension_tether_ground",
    "speed_tangential",
    "distance_radial",
    "angle_of_attack",
]
slider,axes = simulate_pattern(
    run_plots=True,
    path_parameters=PATH_PARAMETERS,
    radial_parameters=RADIAL_PARAMETERS,
    physical_config=PHYSICAL_CONFIG,
    wind_config=WIND_CONFIG,
    variables_to_plot=variables_to_plot,
)
